# Módulo 1 · Plataforma, cómputo y notebooks

⏱ **10 min** · Slides: Módulo 1 (Lakehouse, Azure Databricks, Clusters, Notebooks)

**Objetivo:** ver funcionando lo que los slides describen, antes de tocar datos.

| Paso | Qué haces | Slide |
|---|---|---|
| 1 | Ubicarte en el lakehouse (catálogo → esquema → tabla) | 6–8 |
| 2 | Mezclar Python y SQL con *magic commands* | 16 |
| 3 | Mirar el cómputo que te asignaron | 9–15 |
| 4 | Ver las particiones (lo que reparte el driver) | 12 |
| 5 | Comprobar que el estado vive en la sesión | 17 |
| 6 | Parametrizar con widgets | 18 |
| 7 | Manejo de secretos | 18 |

## Paso 0 · Cargar el setup
`%run` ejecuta otro notebook y trae sus variables y funciones. Debe ir **solo** en su celda.

In [0]:
%run ./00_Setup

## Paso 1 · ¿Dónde estoy? Unity Catalog

Todo dato se nombra con **tres niveles**: `catálogo.esquema.tabla`. Es la capa de gobierno del lakehouse.

In [0]:
%sql
SELECT current_user() AS usuario,
       current_catalog() AS catalogo, 
       current_schema() AS esquema

In [0]:
display(spark.sql("SHOW CATALOGS"))

In [0]:
display(spark.sql(f"SHOW SCHEMAS IN `{CATALOG}`"))

👀 **Qué observar:** ahí está `lab_tvmaze`, el esquema que creó el setup.
Abre **Catalog** en la barra izquierda → `workspace` → `lab_tvmaze` → volumen `datos`. Todavía está vacío.

## Paso 2 · Magic commands: Python y SQL en el mismo notebook

Python crea los datos; SQL los consulta. Es la misma información vista desde dos lenguajes.

In [0]:
spark.range(1, 6).withColumnRenamed("id", "n").createOrReplaceTempView("numeros")

In [0]:
%sql
SELECT 
n, 
n * n AS cuadrado FROM numeros

👀 **Qué observar:** la tabla `numeros` nació en una celda Python y se leyó en una celda `%sql`.
Esta misma celda de texto es un `%md`.

## Paso 3 · El cómputo que te asignaron

En el curso verás clusters (driver + workers). En la **Free Edition** el cómputo es **serverless**:
Databricks lo administra por ti.

| | Slides (Azure) | Tu laboratorio (Free Edition) |
|---|---|---|
| Tipo de cómputo | All-purpose / Job cluster | Serverless |
| Configurar nodos, autoscaling | Tú eliges | Lo decide Databricks |
| Auto-terminación | La configuras | Automática |
| Cobro | DBU + VMs | Cuota diaria gratuita |
| Plano de control | Databricks | Databricks (igual) |
| Almacenamiento | ADLS Gen2 (tu suscripción) | Administrado por Databricks |

In [0]:
import os

print("Versión de Spark :", spark.version)
print("CPUs del driver  :", os.cpu_count())

## Paso 4 · Particiones: el trabajo que se reparte

Spark divide los datos en **particiones**. Cada una es una tarea que un worker puede ejecutar en paralelo.
`spark_partition_id()` te dice a qué partición pertenece cada fila.

In [0]:
df = spark.range(0, 2_000_000)

print(df.count())

In [0]:
por_particion = (df.withColumn("particion", F.spark_partition_id())
                   .groupBy("particion").count()
                   .orderBy("particion"))
display(por_particion)

In [0]:
# Nosotros también podemos pedir más o menos particiones
k2 = (df.repartition(2)
        .select(F.spark_partition_id().alias("p"))
        .distinct().count())
print(f"repartition(2) -> {k2} particiones")

k8 = (df.repartition(8)
        .select(F.spark_partition_id().alias("p"))
        .distinct().count())
print(f"repartition(8) -> {k8} particiones")

👀 **Qué observar:** el número de particiones cambia, pero el resultado de los datos no.
Más particiones = más trabajo en paralelo (siempre que haya workers para ejecutarlas).

## Paso 5 · El estado vive en la sesión

Variables y DataFrames existen en la memoria del cómputo, no en el notebook.

In [0]:
mi_variable = "Vivo en la memoria de esta sesión"
print(mi_variable)

🧪 **Pruébalo:** desconecta y vuelve a conectar el cómputo (selector arriba a la derecha) y ejecuta
solo la celda de abajo.

In [0]:
print(mi_variable)

👀 **Qué observar:** `NameError`. La variable se perdió con la sesión.
Por eso cada módulo empieza con `%run ./00_Setup`, y por eso se re-ejecuta de arriba hacia abajo.

## Paso 6 · Widgets: parámetros del notebook

Un widget convierte un valor fijo en un parámetro. Los Jobs usan este mecanismo para pasar datos.

In [0]:
dbutils.widgets.text("nombre", "Estudiante", "Tu nombre")
print("Hola,", dbutils.widgets.get("nombre"))

🧪 **Pruébalo:** escribe tu nombre en el widget (barra superior del notebook) y vuelve a ejecutar la celda anterior.

## Paso 7 · Secretos: nunca en el código

❌ **Mal:** `api_key = "sk_abc123..."` (queda en el notebook, en el historial y en Git).

✅ **Bien:** pedirla a un *secret scope* (por ejemplo respaldado por Azure Key Vault):
```python
api_key = dbutils.secrets.get(scope="kv", key="token")
```
Databricks oculta el valor si intentas imprimirlo (`[REDACTED]`).

In [0]:
try:
    print(dbutils.secrets.listScopes())
except Exception as e:
    print("No hay secret scopes disponibles en este workspace:", type(e).__name__)

## ✔ Checkpoint

- [ ] Ubico `catálogo.esquema.tabla` en Catalog
- [ ] Sé qué hace `%sql`, `%md` y `%run`
- [ ] Entiendo que el trabajo se reparte en particiones
- [ ] Sé que el estado se pierde al desconectar
- [ ] Uso widgets y secretos en lugar de valores fijos

**Siguiente:** `02_Ingesta_RAW`